In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Importing Dataset and Libraries


In [2]:

import matplotlib.pyplot as plt
import numpy as np
import os
import PIL
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.python.keras.layers import Dense, Flatten
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

In [3]:
import zipfile

In [5]:
zip_ref = zipfile.ZipFile('/content/drive/MyDrive/datasets/train.zip','r')
zip_ref.extractall('/tmp')
zip_ref.close()

In [6]:
zip_ref = zipfile.ZipFile('/content/drive/MyDrive/datasets/valid.zip','r')
zip_ref.extractall('/tmp')
zip_ref.close()

In [7]:
zip_ref = zipfile.ZipFile('/content/drive/MyDrive/datasets/test.zip','r')
zip_ref.extractall('/tmp')
zip_ref.close()

In [8]:
# prompt: different class names

base_dir = '/tmp/train/'
class_names = os.listdir(base_dir)
print(class_names)


['acne', 'oil', 'dry']


#Model Building

In [ ]:
from tensorflow.keras.layers import Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
# Define the Sequential model
keras_model = Sequential()

# Add the pre-trained DenseNet169 model
pretrained_model = tf.keras.applications.DenseNet169(include_top=False,
                   input_shape=(640, 640, 3),
                   pooling='avg', weights='imagenet')

# Freeze the layers of the pre-trained model
for layer in pretrained_model.layers:
    layer.trainable = False

keras_model.add(pretrained_model)
keras_model.add(Flatten())

# Add a dense layer with dropout and batch normalization
keras_model.add(Dense(512, activation='relu'))
keras_model.add(BatchNormalization())
keras_model.add(Dropout(0.5))

# Add another dense layer
keras_model.add(Dense(256, activation='relu'))
keras_model.add(BatchNormalization())
keras_model.add(Dropout(0.5))

# Add the final classification layer
keras_model.add(Dense(3, activation='softmax'))

# Compile the model
keras_model.compile(optimizer=Adam(learning_rate=0.001),
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])

# Print the model summary
keras_model.summary()


51877672/51877672 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
keras_model.summary()

In [ ]:
keras_model.compile(optimizer=Adam(learning_rate=0.001),loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define data generators for training, validation, and test sets
train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
        '/tmp/train/',
        target_size=(640, 640),
        batch_size=32,
        class_mode='categorical')

valid_datagen = ImageDataGenerator(rescale=1./255)
validation_generator = valid_datagen.flow_from_directory(
        '/tmp/valid/',
        target_size=(640, 640),
        batch_size=32,
        class_mode='categorical')

test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
        '/tmp/test/',
        target_size=(640, 640),
        batch_size=32,
        class_mode='categorical')

# Fit the model using the generators
history = keras_model.fit(
      train_generator,
      steps_per_epoch=train_generator.samples/train_generator.batch_size,
      validation_data=validation_generator,
      validation_steps=validation_generator.samples/validation_generator.batch_size,
      epochs=20)

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.grid()
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend(['train', 'validation'])
plt.show()

In [ ]:
#plt.figure(figsize=(10, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.grid()
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epochs')
plt.legend(loc='best')
plt.show()


#Testing the model

In [ ]:

test_loss, test_acc = keras_model.evaluate(test_generator, verbose=2)
print('\nTest accuracy from evaluate:', test_acc)


In [ ]:
test_loss, test_acc = keras_model.evaluate(test_generator, verbose=2)
print('\nTest accuracy:', test_acc)

In [ ]:
import math

# Evaluate the model using the test generator
steps = math.ceil(test_generator.samples / test_generator.batch_size)
test_loss, test_accuracy = keras_model.evaluate(test_generator, steps=steps)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Make predictions on the test set
test_generator.reset()  # Reset the generator to start from the beginning
predictions = keras_model.predict(test_generator, steps=steps, verbose=1)

# Get the class indices and labels
class_indices = test_generator.class_indices
labels = list(class_indices.keys())

# Function to decode predictions
def decode_predictions(predictions, labels, top=5):
    decoded = []
    for pred in predictions:
        top_indices = pred.argsort()[-top:][::-1]
        top_labels = [labels[i] for i in top_indices]
        top_scores = [pred[i] for i in top_indices]
        decoded.append(list(zip(top_labels, top_scores)))
    return decoded

# Decode the predictions
decoded_predictions = decode_predictions(predictions, labels)

# Print the predictions for the first few test images
for i in range(min(len(decoded_predictions), 5)):
    print(f"Test Image {i+1}:")
    for label, score in decoded_predictions[i]:
        print(f"   Label: {label}, Confidence: {score:.4f}")


#Prediction from real data

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

# Function to load and preprocess a single image
def load_and_preprocess_image(img_path, target_size=(640, 640)):
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    img_array = img_array / 255.0  # Rescale to [0, 1] if you used rescale=1./255 in ImageDataGenerator
    return img_array

# Function to predict the class of a single image
def predict_image_class(model, img_path, class_indices):
    # Load and preprocess the image
    img_array = load_and_preprocess_image(img_path)

    # Make prediction
    predictions = model.predict(img_array)

    # Get the class labels
    labels = list(class_indices.keys())

    # Decode the prediction
    top_index = np.argmax(predictions[0])
    top_label = labels[top_index]
    top_confidence = predictions[0][top_index]

    return top_label, top_confidence

# Assuming class indices from the test generator
class_indices = {'acne': 0, 'dry': 1, 'oily': 2}

# Path to the image you want to predict
test_image_path = '/content/drive/MyDrive/Datasets/dry.jpeg'  # Replace with your test image path

# Predict the class of the test image
predicted_label, confidence = predict_image_class(keras_model, test_image_path, class_indices)

# Print the prediction
print(f"The image belongs to: {predicted_label} with confidence: {confidence:.4f}")

# Optionally, display the image
img = image.load_img(test_image_path, target_size=(640, 640))
plt.imshow(img)
plt.title(f"Predicted: {predicted_label} ({confidence:.4f})")
plt.axis('off')
plt.show()


In [ ]:
model_version=1
keras_model.save('/content/drive/MyDrive/model')

#Skincare recommendation from data obtained through webscrapping

In [ ]:
import pandas as pd

In [ ]:
import pandas as pd

acne_products = pd.read_csv('/content/drive/MyDrive/Datasets/Acne1.csv')
oily_products = pd.read_csv('/content/drive/MyDrive/Datasets/Oily1.csv')
dry_products = pd.read_csv('/content/drive/MyDrive/Datasets/Dry1.csv')


In [ ]:
oily = oily_products.sort_values(by='Rating', ascending=False)
dry = dry_products.sort_values(by='Rating', ascending=False)
acne = acne_products.sort_values(by='Rating', ascending=False)

In [ ]:
from IPython.display import display, HTML

def make_clickable(link):
    return f'<a href="{link}" target="_blank">{link}</a>'

oily['Product_Link'] = oily['Product_Link'].apply(make_clickable)
dry['Product_Link'] = dry['Product_Link'].apply(make_clickable)
acne['Product_Link'] = acne['Product_Link'].apply(make_clickable)


In [ ]:
def display_recommended_products(recommended_products,width='100%'):
    #display(HTML(recommended_products.to_html(escape=False)))
    html=recommended_products[['Product_Name', 'Rating', 'Product_Link', 'Description','Price']].to_html(escape=False, index=False)
    styled_html = f'<div style="width: {width}; margin: 0 auto;">{html}</div>'
    display(HTML(styled_html))


def recommend_products(predicted_label, num_recommendations=5):
    if predicted_label == 'acne':
        recommended_products = acne_products.head(num_recommendations)
    elif predicted_label == 'oily':
        recommended_products = oily_products.head(num_recommendations)
    elif predicted_label == 'dry':
        recommended_products = dry_products.head(num_recommendations)
    else:
        raise ValueError("Unknown skin type predicted.")
    return recommended_products


recommended_products = recommend_products(predicted_label)
recommended_products['Product_Link'] = recommended_products['Product_Link'].apply(make_clickable)
display_recommended_products(recommended_products,width='100%')